In [ ]:
# Imports

import os
import time
from collections.abc import Iterator

import gradio as gr
from dotenv import load_dotenv
from openai import OpenAI
from openai.types.chat import (
    ChatCompletionMessageParam as Message,
    ChatCompletionSystemMessageParam as SystemMessage,
    ChatCompletionUserMessageParam as UserMessage,
)

In [ ]:
# One key, one client — OpenRouter speaks the OpenAI protocol, so the OpenAI SDK is the client
# for every model below. override=True lets an edited .env win over a stale shell export.
load_dotenv(override=True)

api_key = os.getenv("OPENROUTER_API_KEY")
if not api_key:
    raise RuntimeError(
        "OPENROUTER_API_KEY is not set. Put it in the .env file at the project root "
        "(keys live at https://openrouter.ai/keys) and rerun this cell."
    )
print(f"OpenRouter key loaded, begins {api_key[:8]}")

client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=api_key)

In [ ]:
# The dropdown: label shown in the UI → OpenRouter model ID. Every one of these ships open
# weights. Any other OpenRouter ID works too — "qwen/qwen3-coder-next", "anthropic/claude-opus-5"
# — and shows up in the dropdown the next time the app cell runs.

MODELS = {
    "DeepSeek V4 Flash": "deepseek/deepseek-v4-flash",
    "DeepSeek V4 Pro": "deepseek/deepseek-v4-pro",
    "MiniMax M3": "minimax/minimax-m3",
    "MiMo-V2.5": "xiaomi/mimo-v2.5",
    "GLM 5.2": "z-ai/glm-5.2",
    "Kimi K3": "moonshotai/kimi-k3",
    "Hy3": "tencent/hy3",
    "Nemotron 3 Ultra (free)": "nvidia/nemotron-3-ultra-550b-a55b:free",
}
DEFAULT_MODEL = "DeepSeek V4 Flash"

# All of these think before answering, and the thinking is billed as output tokens. OpenRouter's
# unified `reasoning` block is translated into whatever each provider calls the same knob, so one
# dropdown covers models that disagree about how to spell it.
EFFORTS = ["low", "medium", "high"]
DEFAULT_EFFORT = "medium"

# Generous, because on a reasoning model this ceiling covers the thinking as well as the answer,
# and a port that runs out of budget stops mid-function. You are billed for tokens produced,
# not for the ceiling.
MAX_TOKENS = 32_000

In [ ]:
SYSTEM_PROMPT = """You port Python programs to Rust.

Reply with the contents of a single Rust source file and nothing else: no prose, no
explanation, no Cargo.toml, no shell commands. Use the standard library only — no external
crates. Comments inside the code are welcome wherever a choice is not obvious.

The program must print exactly what the Python program prints — same values, same
formatting, same number of lines — in as little wall-clock time as possible."""


def port_messages(python: str) -> list[Message]:
    """The two turns of the conversation: the standing rules, and the program to port."""
    return [
        SystemMessage(role="system", content=SYSTEM_PROMPT),
        UserMessage(
            role="user",
            content=f"Port this Python program to Rust:\n\n```python\n{python}\n```",
        ),
    ]

In [ ]:
# What sits in the Rust box before there is a port. It is never left truly empty: an empty
# gr.Code renders a placeholder panel instead of an editor, and that panel has a height of its
# own, so the two sides would not match until the first port arrived.
EMPTY_PORT = "// The Rust port appears here."


def unfence(reply: str) -> str:
    """Drop Markdown fences: asked point blank for a bare source file, models wrap it anyway."""
    lines = [line for line in reply.splitlines() if not line.lstrip().startswith("```")]
    return "\n".join(lines).strip()


def convert(
    python: str, label: str, effort: str, progress: gr.Progress = gr.Progress()
) -> Iterator[tuple[str, str]]:
    """Stream `label`'s Rust port of `python`, yielding (code so far, status line).

    A reasoning model can be quiet for minutes before the first line of code: the thinking
    arrives on `delta.reasoning`, not `delta.content`, so the code box stays empty while the
    model works. Both the status line and the progress bar count what has arrived so far —
    neither total is knowable in advance, so the bar is fed `(count, None)`, a counter that
    moves rather than a bar that fills.

    Repainting on every chunk would make the browser, not the model, the bottleneck, so the
    boxes are updated five times a second and once more when the stream ends.
    """
    stream = client.chat.completions.create(
        model=MODELS[label],
        messages=port_messages(python),
        max_tokens=MAX_TOKENS,
        extra_body={"reasoning": {"effort": effort}},
        stream=True,
    )
    started = time.monotonic()
    last_paint = 0.0
    reply = ""
    thought = 0
    for chunk in stream:
        # A choice-less chunk is a keep-alive or the closing usage record.
        if not chunk.choices:
            continue
        delta = chunk.choices[0].delta
        thought += len((delta.model_extra or {}).get("reasoning") or "")
        reply += delta.content or ""
        elapsed = time.monotonic() - started
        if elapsed - last_paint < 0.2:
            continue
        last_paint = elapsed
        if reply:
            rust = unfence(reply)
            progress((len(rust), None), desc=f"{label} is writing", unit="characters")
            yield rust, f"{label} is writing — {len(rust):,} characters, {elapsed:.0f}s"
        else:
            progress((thought, None), desc=f"{label} is thinking", unit="characters")
            yield (
                EMPTY_PORT,
                f"{label} is thinking — {thought:,} characters, {elapsed:.0f}s",
            )

    elapsed = time.monotonic() - started
    yield (
        unfence(reply),
        f"{label} finished in {elapsed:.0f}s — {thought:,} characters of thinking",
    )

In [ ]:
# Three programs to start from, and a place to paste your own.
#
# HELLO is the default because it is the cheapest way to see the whole loop work: a few hundred
# tokens, seconds rather than minutes, and an answer you can check at a glance.
#
# PI is a pure measurement of what an interpreted loop costs: no traps, no cleverness available
# beyond what the optimiser finds on its own.
#
# MAX_SUBARRAY is the opposite — three traps in twenty lines. The generator multiplies a 32-bit
# state by 1664525, which overflows u32, so a port that picks the obvious width compiles cleanly
# and prints a different number. The output is formatted with `{total:,}`, which Rust cannot do
# at all. And the quadratic scan is the kind of loop a model is tempted to replace with Kadane's
# algorithm, which is a different program that happens to agree on this input.

HELLO = """
print("Hello, world!")
"""

PI = """
import time


def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations + 1):
        j = i * param1 - param2
        result -= 1 / j
        j = i * param1 + param2
        result += 1 / j
    return result


start_time = time.time()
result = calculate(100_000_000, 4, 1) * 4
end_time = time.time()

print(f"Result: {result:.12f}")
print(f"Execution Time: {(end_time - start_time):.6f} seconds")
"""

MAX_SUBARRAY = """
import time


def lcg(seed):
    value = seed
    while True:
        value = (1664525 * value + 1013904223) % 2**32
        yield value


def random_numbers(count, seed, low, high):
    generator = lcg(seed)
    return [next(generator) % (high - low + 1) + low for _ in range(count)]


def max_subarray_sum(numbers):
    best = numbers[0]
    for i in range(len(numbers)):
        total = 0
        for j in range(i, len(numbers)):
            total += numbers[j]
            if total > best:
                best = total
    return best


start_time = time.time()
seeds = lcg(42)
total = 0
for _ in range(10):
    total += max_subarray_sum(random_numbers(2_000, next(seeds), -10, 10))
end_time = time.time()

print(f"Total maximum subarray sum (10 runs): {total:,}")
print(f"Execution Time: {(end_time - start_time):.6f} seconds")
"""

EXAMPLES = {
    "hello world": HELLO,
    "pi — a convergent series": PI,
    "maximum subarray sum": MAX_SUBARRAY,
}
DEFAULT_EXAMPLE = "hello world"

In [ ]:
# Two things Gradio's code editor gets wrong for this app, both fixed in CSS at launch.
#
# It draws at --text-sm (12px) in --body-text-color, a mid grey that reads as washed out on a
# white background. The first three rules darken text that has no syntax colour of its own and
# bump the size; the highlighting itself is untouched, and the dark-mode line keeps the boxes
# readable if the browser asks Gradio for its dark theme.
#
# And it sizes each editor from its own content: `lines` and `max_lines` are turned into a
# min-height and max-height by measuring a rendered line *in that box*, so a box holding one
# line and a box holding thirty settle at different heights no matter what those props say.
# Pinning the scroller to one height overrides the measurement — both boxes are always this
# tall, and anything longer scrolls inside.
CSS = """
.cm-editor { font-size: var(--text-md); }
.cm-editor .cm-content, .cm-editor .cm-line { color: var(--neutral-950); }
.dark .cm-editor .cm-content, .dark .cm-editor .cm-line { color: var(--neutral-50); }
.cm-editor .cm-scroller { min-height: 30rem !important; max-height: 30rem !important; }
"""

# Roughly the 30rem above, and what the boxes fall back to if the stylesheet ever stops matching.
BOX_LINES = 25


def build_demo() -> gr.Blocks:
    """Build the UI: Python in on the left, a streamed Rust port out on the right."""

    def load_example(name: str) -> tuple[str, str, str]:
        """Swap in a different program, clearing the port of the old one."""
        return EXAMPLES[name], EMPTY_PORT, ""

    with gr.Blocks(title="llm-engineering — Python → Rust") as demo:
        gr.Markdown("# Python → Rust\nPick an open-weight model and hand it a program.")

        with gr.Row(equal_height=True):
            python_box = gr.Code(
                label="Python",
                language="python",
                value=EXAMPLES[DEFAULT_EXAMPLE],
                lines=BOX_LINES,
                max_lines=BOX_LINES,
                scale=1,
            )
            # Gradio's code editor has no Rust lexer, so this box is deliberately unlit.
            rust_box = gr.Code(
                label="Rust",
                language=None,
                value=EMPTY_PORT,
                lines=BOX_LINES,
                max_lines=BOX_LINES,
                scale=1,
            )
        with gr.Row():
            example = gr.Dropdown(
                choices=list(EXAMPLES), value=DEFAULT_EXAMPLE, label="Example program"
            )
            model = gr.Dropdown(
                choices=list(MODELS), value=DEFAULT_MODEL, label="Model"
            )
            effort = gr.Dropdown(
                choices=EFFORTS, value=DEFAULT_EFFORT, label="Reasoning effort"
            )
        convert_button = gr.Button("Convert to Rust", variant="primary")
        # The bar Gradio draws from progress() lives on the components being written to, and
        # vanishes the moment the first token lands in them. This line is the one thing on
        # screen that keeps saying what is happening for as long as the model is thinking.
        status = gr.Textbox(label="Status", interactive=False)

        example.change(
            load_example, inputs=example, outputs=[python_box, rust_box, status]
        )
        convert_button.click(
            convert, inputs=[python_box, model, effort], outputs=[rust_box, status]
        )

    return demo

In [ ]:
# Serves until you interrupt the kernel, or run demo.close() in a new cell. Gradio 6 takes
# styling at launch rather than on the Blocks, so the stylesheet is handed over here.

demo = build_demo()
demo.launch(css=CSS)